# Chapter 17: Putting It All Together

[Read this chapter online](https://jackluu.io/book/section-5-generation/ch17-putting-it-all-together/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch17-putting-it-all-together.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 17: Putting It All Together

![You are here in the big picture](../assets/diagrams/ch17-where-we-are.png){ width="756" }
*Figure 17.1: Where we are: completing the full pipeline from text to language model.*

You have built a language model from zero. Step by step, you wrote the code to tokenize text, embed it into vectors, apply self-attention, stack transformer blocks, train the weights using gradient descent, and generate new text with temperature and top-k sampling. In this final chapter you will:

- Run the complete end-to-end pipeline in a single script.
- See the full architecture in action.
- Understand how your model relates to modern, production-grade LLMs.

**Words to Know**
    - **End to End**: a process that takes raw input (text) and goes through every necessary step to produce the final output (a trained model and generated text) without manual intervention.
    - **Instruction tuning**: A post-training step that teaches the model to answer questions and follow instructions.
    - **RLHF**: Reinforcement Learning from Human Feedback, a method to train models to behave politely and align with human preferences.

## Theory

### The Full Architecture

Take a look back at everything you built. This is not a "toy" architecture. You just wrote the exact same building blocks used by GPT-2, which is the architectural foundation of GPT-3, GPT-4, and many other modern Large Language Models (LLMs).

![The final full system map](../assets/diagrams/ch17-putting-it-all-together.png){ width="756" }
*Figure 17.2: The complete system: from raw text to a trained language model.*

The differences between your model and a massive production model are mostly a matter of scale:

- **Vocabularies**: We used 65 characters; they use 50,000+ subword tokens.
- **Dimensions**: We used a 128-dimensional embedding and 4 layers; they use thousands of dimensions and nearly a hundred layers.
- **Data**: We trained on 1 megabyte of Shakespeare; they train on terabytes of internet text.
- **Hardware**: We trained for a few minutes on a CPU; they train for months on thousands of specialized GPUs.

### How Real Models Differ

While the fundamental transformer architecture (embeddings, attention, blocks, training) remains the same, modern models add refinements to optimize performance:

- Instead of simple position lookups, they might use *Rotary Positional Embeddings*, which help models understand relative distances between words better.
- Instead of standard multi-head attention, they might use *Grouped-Query Attention*, which saves memory and speeds up text generation.
- After pretraining (what we did), they undergo *Instruction Tuning* and *RLHF (Reinforcement Learning from Human Feedback)* to learn how to answer questions politely rather than just predicting the next word.

As the "where we are" map shows, the end-to-end pipeline connects every stage from text ingestion to generation, wrapping the entire system into one continuous flow.

**In Business**
    For our house-style assistant, this end-to-end script represents the full product lifecycle. In a business environment, you run this pipeline whenever the company archive changes: training a new model overnight on updated documents, validating it, and deploying the new checkpoint to serve your marketing and compliance teams.

## Code

We have combined every piece of code from the previous chapters into one master script. It downloads the data, tokenizes it, creates the DataLoaders, builds the 825,000-parameter model, trains it for 3,000 steps, saves a checkpoint, and generates a sample text. 

![Code flow: Raw Text -> Tokenize -> Train Model -> Save .pt -> Generate](../assets/diagrams/ch17-code-flow.png){ width="678" }
*Figure 17.3: The full script executes every step in sequence without manual intervention.*

Let's run the smoke test.

```python
$ python src/ch16_full_pipeline.py
[1/7] Downloading dataset...
...
[2/7] Tokenizing...
...
[3/7] Creating DataLoaders...
...
[4/7] Building model...
...
[5/7] Training for 3000 steps...
...
step     1 | train: 4.2726 | val: 4.1660 | elapsed: 2s | ETA: 6148s
...
step  3000 | train: 1.7323 | val: 1.7411 | elapsed: 474s | ETA: 0s
...
[6/7] Saving checkpoint...
...
[7/7] Generating text...
...
GENERATED TEXT (temperature=0.8, top_k=40):
...
ROMEO:
Praviour soul to shall that that are the and not,
```

**What just happened:**

- The model trained successfully and loss steadily decreased. 
- It generated brand new, Shakespeare-like text based on the patterns it learned.
- While it makes some logical or grammatical mistakes, the character names, sentence structures, and vocabulary strongly mimic the training data.

```python
$ python diagrams/charts/ch17_generated_text.py
Saved chart: ch17-generated-text.png
```

![Generated Shakespeare text rendered as an image](../assets/diagrams/ch17-generated-text.png){ width="650" }
*Figure 17.4: The model generates new text.*

### Shape Check

Table 17.1 summarizes the parameter count of the fully assembled network.

**Table 17.1:** Final parameter count of the completed model.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `model parameters` | `824,832` | The total number of weights the model learned during training. |

## Try It

**Try It**
    Try testing the final model like a deployed production service. Create a script to load `model_final.pt` and respond to a user prompt.
    
    ```python title="src/examples/ch17_deploy.py (excerpt)" linenums="1" hl_lines="6 8"
    # ...
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model_final.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        # Fall back to model.pt if model_final.pt does not exist
        checkpoint_path = os.path.join(
            os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
        )
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    # ...
    ```
    
    Lines 6 and 8 check if the final checkpoint exists and fall back to a previous save if it doesn't.
    
    ```console title="Terminal"
    $ python src/examples/ch17_deploy.py
    --- Prompt: 'USER: What is the news?\n' ---
    USER: What is the news?

    FROKENTER:
    And with is there?

    GLOUCEMIO:
    All But, there come from the prainion; and every,
    As do p

    ```


## Key Takeaways

- You successfully built the complete GPT architecture from scratch.
- The fundamental components (tokenization, embeddings, self-attention, transformer blocks, and gradient descent) are the core of all modern LLMs.
- Larger models mostly scale up these exact same components with more data, more parameters, and better hardware.
- Refinements like RLHF and instruction tuning are applied *after* this pretraining process.

## Check Your Understanding

1. What is the difference between the model we built and GPT-2?
2. Why does the model output sometimes contain spelling or logic errors despite being fully trained?
3. What is the purpose of an end-to-end pipeline in a business setting?


## Further Reading

**Where prompting came from.** Even a pre-trained model normally had to be fine-tuned, with fresh labeled examples and a training run, before it could do a new job. At 175 billion parameters the authors found something different: write two or three examples into the prompt and the model follows the pattern, with no weights changed at all. That behavior is what people now call prompting, and it is why a language model became something you talk to rather than something you retrain.

**The step that turns a text predictor into an assistant.** A model trained to continue text is not the same thing as a model that does what you ask; it will happily continue your question with more questions. The authors collected human demonstrations of good answers and human rankings of competing answers, and fine-tuned on that feedback. The result matters for how you read this book: a 1.3 billion parameter model trained this way was preferred by people to the 175 billion parameter model it came from. Capability and helpfulness are different problems, and this book builds the first one.

**Why the field started building bigger.** Before this, deciding how large to make a model, how much text to train it on, and how much compute to spend was guesswork. The paper measured all three and found the error falls along smooth, predictable curves across a very wide range of sizes. That turned model building into a budgeting exercise, and it is the reason the industry spent the following years scaling up. It also explains the ceiling on the model you train here: a few hundred thousand parameters and a few hundred thousand characters of Shakespeare buy a certain quality of text, and no more.

<div class="refs" markdown>

Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan, J., Dhariwal, P., Neelakantan, A., Shyam, P., Sastry, G., Askell, A., Agarwal, S., Herbert-Voss, A., Krueger, G., Henighan, T., Child, R., Ramesh, A., Ziegler, D. M., Wu, J., Winter, C., ... Amodei, D. (2020). *Language models are few-shot learners* (arXiv:2005.14165). arXiv. https://doi.org/10.48550/arXiv.2005.14165

Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., Gray, S., Radford, A., Wu, J., & Amodei, D. (2020). *Scaling laws for neural language models* (arXiv:2001.08361). arXiv. https://doi.org/10.48550/arXiv.2001.08361

Ouyang, L., Wu, J., Jiang, X., Almeida, D., Wainwright, C. L., Mishkin, P., Zhang, C., Agarwal, S., Slama, K., Ray, A., Schulman, J., Hilton, J., Kelton, F., Miller, L., Simens, M., Askell, A., Welinder, P., Christiano, P., Leike, J., & Lowe, R. (2022). *Training language models to follow instructions with human feedback* (arXiv:2203.02155). arXiv. https://doi.org/10.48550/arXiv.2203.02155

*Note.* The Tiny Shakespeare text used throughout the book comes from Andrej Karpathy's char-rnn project: https://github.com/karpathy/char-rnn{ .note }

</div>

---

### `src/ch16_full_pipeline.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch16_full_pipeline.py"   # a cell has none, and the file uses it to find the text

"""
End-to-end pipeline to build and train the LLM.
This file belongs to Chapter 17.
Run: python src/ch16_full_pipeline.py
"""
import os
import time
import requests
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import sys

from src.utils.config import GPTConfig, TrainConfig
from src.ch09_gpt_model import GPT

# Settings
gpt_cfg   = GPTConfig()
train_cfg = TrainConfig()

DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")
DATA_URL  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
CKPT_PATH = os.path.join(train_cfg.checkpoint_dir, "model_final.pt")

# --- The Idea ---

class TextDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx     : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

@torch.no_grad()
def val_loss_estimate(model, val_loader, vocab_size):
    model.eval()
    it = iter(val_loader)
    losses = []
    for _ in range(min(50, len(val_loader))):
        x, y = next(it)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

def generate(
    model, prompt, encode, decode, max_new_tokens=300,
    temperature=0.8, top_k=40
):
    ids = torch.tensor([encode(prompt)], dtype=torch.long)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx    = ids[:, -gpt_cfg.block_size:]
            logits = model(ctx)[:, -1, :] / temperature
            if top_k:
                thresh = logits.topk(top_k).values[:, -1, None]
                logits = logits.masked_fill(logits < thresh, float("-inf"))
            probs   = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            ids     = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

# --- Demo ---
if __name__ == "__main__":
    print("Chapter 17: Building an LLM from Zero -- Full Pipeline\n")

    torch.manual_seed(42)

    print("[1/7] Downloading dataset...")
    if not os.path.exists(DATA_PATH):
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        r = requests.get(DATA_URL)
        r.raise_for_status()
        with open(DATA_PATH, "w", encoding="utf-8") as f:
            f.write(r.text)
        print(f"      Downloaded {os.path.getsize(DATA_PATH)//1024} KB")
    else:
        print(f"      Already exists ({os.path.getsize(DATA_PATH)//1024} KB)")

    print("\n[2/7] Tokenizing...")
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    chars  = sorted(set(text))
    vocab_size = len(chars)
    char_to_id = {ch: i for i, ch in enumerate(chars)}
    id_to_char = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [char_to_id[c] for c in s]
    decode = lambda ids: "".join([id_to_char[i] for i in ids])

    data  = torch.tensor(encode(text), dtype=torch.long)
    n     = len(data)
    split = int(0.9 * n)
    train_data = data[:split]
    val_data   = data[split:]
    print(f"      {n:,} tokens, vocab={vocab_size}, "
          f"train={len(train_data):,}, val={len(val_data):,}")

    print("\n[3/7] Creating DataLoaders...")

    train_loader = DataLoader(
        TextDataset(train_data, gpt_cfg.block_size),
        batch_size=train_cfg.batch_size, shuffle=True
    )
    val_loader   = DataLoader(
        TextDataset(val_data,   gpt_cfg.block_size),
        batch_size=train_cfg.batch_size, shuffle=False
    )
    train_iter   = iter(train_loader)
    print(f"      {len(train_loader):,} train batches, "
          f"{len(val_loader):,} val batches")

    print("\n[4/7] Building model...")
    model     = GPT(gpt_cfg)
    n_params  = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg.learning_rate)
    print(f"      {n_params:,} parameters")

    print(f"\n[5/7] Training for {train_cfg.max_iters} steps...")
    print(f"      Logging every {train_cfg.eval_interval} steps\n")

    start = time.time()
    recent_losses = []
    val_loss = float("inf")

    for step in range(1, train_cfg.max_iters + 1):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        logits = model(x)
        loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        recent_losses.append(loss.item())

        if step % train_cfg.eval_interval == 0 or step == 1:
            val_loss  = val_loss_estimate(model, val_loader, vocab_size)
            recent = recent_losses[-train_cfg.eval_interval:]
            avg_train = sum(recent) / len(recent)
            elapsed   = time.time() - start
            eta       = (elapsed / step) * (train_cfg.max_iters - step)
            print(f"step {step:5d} | train: {avg_train:.4f} | "
                  f"val: {val_loss:.4f} | "
                  f"elapsed: {elapsed:.0f}s | ETA: {eta:.0f}s")

    total_time = time.time() - start
    print(f"      Training done in {total_time:.0f}s")

    print("\n[6/7] Saving checkpoint...")
    os.makedirs(train_cfg.checkpoint_dir, exist_ok=True)
    torch.save({
        "model_state": model.state_dict(),
        "gpt_cfg"    : gpt_cfg,
        "step"       : train_cfg.max_iters,
        "val_loss"   : val_loss,
    }, CKPT_PATH)
    print(f"      Saved to: {CKPT_PATH}")

    print("\n[7/7] Generating text...")
    model.eval()

    print("\nGENERATED TEXT (temperature=0.8, top_k=40):\n")
    print(generate(model, "ROMEO:\n", encode, decode, max_new_tokens=400))
    print("\nCongratulations! You just built and trained an LLM from zero.")

---

### `src/examples/ch17_deploy.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch17_deploy.py"   # a cell has none, and the file uses it to find the text

"""Load the model and generate text for a new prompt."""
import os
import sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode, decode
from src.ch09_gpt_model import GPT
from src.ch15_generate_sampling import generate

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model_final.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        # Fall back to model.pt if model_final.pt does not exist
        checkpoint_path = os.path.join(
            os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
        )
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    
    text_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "src", "data", "shakespeare.txt"
    )
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()
    chars, char_to_id, id_to_char = build_vocab(text)
    
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    prompt = "USER: What is the news?\n"
    print(f"--- Prompt: {repr(prompt)} ---")
    
    output = generate(
        model, prompt, encode_fn, decode_fn, cfg, 
        max_new_tokens=100, temperature=0.8, top_k=40
    )
    print(output)

if __name__ == "__main__":
    main()